In [14]:
%pip install -q google-genai

StatementMeta(, d122b542-f558-4572-b018-72c797b161e7, 30, Finished, Available, Finished, True)


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /nfs4/pyenv-9d22cdd7-da65-42b9-9b48-693acbb439c4/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



In [15]:
import os, json, random, time
import numpy as np
from PIL import Image
from google import genai

api_key = notebookutils.fs.head("Files/config/gemini_api_key.txt").strip()
client = genai.Client(api_key=api_key)

MODELO = "gemini-3.5-flash-lite"

# v2: en la v1 el modelo sobrestimaba ~13 puntos de media porque contaba
# también los pasillos entre filas. Le explico qué debe contar y qué no.
VERSION_PROMPT = "v2"
PROMPT = """Esta es una imagen aérea (ortofoto) vista desde arriba, de unos 300 x 300 metros.
1) Clasifica la parcela en una de estas fases de obra:
   terreno_sin_tocar, movimiento_de_tierras, estructuras_sin_paneles, paneles_instalados.
2) Estima qué porcentaje de la imagen ocupa la superficie de los paneles solares.
   Cuenta SOLO la superficie de las placas (los módulos oscuros).
   NO cuentes los pasillos de tierra entre filas, caminos, vallas, edificios
   ni el resto del recinto de la planta.
   Muchas imágenes tienen pocos paneles o ninguno; si no hay, responde 0.
Responde solo con un JSON así: {"fase": "...", "pct_paneles": 0-100}"""

N_MUESTRAS = 50
PAUSA_SEG = 5
TABLA = "Lh_Gold.validacion_gemini"

StatementMeta(, d122b542-f558-4572-b018-72c797b161e7, 32, Finished, Available, Finished, False)

In [16]:
carpeta = "/lakehouse/default/Files/datasets/jiang_pv"
todas = []
for raiz, _, archivos in os.walk(carpeta):
    for a in archivos:
        if a.endswith(".bmp") and not a.endswith("_label.bmp"):
            todas.append(os.path.join(raiz, a))

random.seed(42)
muestra = random.sample(sorted(todas), N_MUESTRAS)

# Proceso incremental: no repito imágenes ya validadas con este modelo y este prompt.
# Las filas antiguas no tienen la columna version_prompt (eran la v1).
hechas = set()
if spark.catalog.tableExists(TABLA):
    t = spark.table(TABLA)
    if "version_prompt" in t.columns:
        hechas = {r.imagen for r in t
                  .filter(f"modelo = '{MODELO}' AND version_prompt = '{VERSION_PROMPT}' "
                          "AND pct_gemini IS NOT NULL")
                  .select("imagen").collect()}
pendientes = [m for m in muestra if os.path.basename(m) not in hechas]
print(f"Muestra: {len(muestra)} | Ya hechas: {len(hechas)} | Pendientes: {len(pendientes)}")

StatementMeta(, d122b542-f558-4572-b018-72c797b161e7, 33, Finished, Available, Finished, False)

Muestra: 50 | Ya hechas: 0 | Pendientes: 50


In [17]:
def preguntar_a_gemini(imagen):
    """Llama a Gemini. Si el servidor está saturado (503) espero y reintento."""
    for intento in range(3):
        try:
            r = client.models.generate_content(
                model=MODELO,
                contents=[imagen, PROMPT],
                config={"response_mime_type": "application/json"},
            )
            return json.loads(r.text)
        except Exception as e:
            if "503" in str(e) and intento < 2:
                print("  Servidor saturado, espero 30 s...")
                time.sleep(30)
            else:
                raise

resultados = []
for i, ruta_img in enumerate(pendientes, 1):
    mascara = np.array(Image.open(ruta_img.replace(".bmp", "_label.bmp")).convert("L"))
    pct_real = float((mascara > 0).mean() * 100)

    try:
        resp = preguntar_a_gemini(Image.open(ruta_img).convert("RGB"))
    except Exception as e:
        # Si se acaba la cuota no tiene sentido seguir: lo que falte se hace mañana
        if "429" in str(e):
            print("Cuota agotada, paro aquí. Lo pendiente se procesará en la siguiente ejecución.")
            break
        print(f"  Error en {os.path.basename(ruta_img)}: {str(e)[:100]}")
        continue

    resultados.append({
        "imagen": os.path.basename(ruta_img),
        "tipo_terreno": os.path.basename(os.path.dirname(ruta_img)),
        "pct_real": round(pct_real, 2),
        "pct_gemini": float(resp["pct_paneles"]),
        "fase_gemini": resp["fase"],
        "error_api": None,
    })
    print(f"{i}/{len(pendientes)}  real={pct_real:5.1f}%  gemini={resp['pct_paneles']}")
    time.sleep(PAUSA_SEG)

print(f"Nuevas validaciones correctas: {len(resultados)}")

StatementMeta(, d122b542-f558-4572-b018-72c797b161e7, 34, Finished, Available, Finished, False)

1/50  real= 10.7%  gemini=14
2/50  real= 81.4%  gemini=80
3/50  real=  0.0%  gemini=0
4/50  real= 10.6%  gemini=8
5/50  real= 41.5%  gemini=52
6/50  real= 21.4%  gemini=45
7/50  real= 76.1%  gemini=75
8/50  real=  4.0%  gemini=4
9/50  real= 53.4%  gemini=52
10/50  real=  7.7%  gemini=14
11/50  real= 12.7%  gemini=14
12/50  real= 50.6%  gemini=52
13/50  real= 20.6%  gemini=45
14/50  real= 77.9%  gemini=52
15/50  real= 47.7%  gemini=55
16/50  real=  2.8%  gemini=2
17/50  real=  2.2%  gemini=2
18/50  real= 76.8%  gemini=75
19/50  real= 31.0%  gemini=23
20/50  real= 15.6%  gemini=23
21/50  real= 58.2%  gemini=45
22/50  real= 77.2%  gemini=75
23/50  real= 10.3%  gemini=14
24/50  real= 42.8%  gemini=56
25/50  real= 21.6%  gemini=35
26/50  real=  1.4%  gemini=5
27/50  real= 33.2%  gemini=27
28/50  real= 44.2%  gemini=52
29/50  real= 75.9%  gemini=75
30/50  real=  1.0%  gemini=1
31/50  real= 17.4%  gemini=14
32/50  real= 40.2%  gemini=62
33/50  real= 57.4%  gemini=52
34/50  real= 35.3%  gemini

In [18]:
from pyspark.sql import functions as F

esquema = ("imagen string, tipo_terreno string, pct_real double, "
           "pct_gemini double, fase_gemini string, error_api string")

if resultados:
    nuevos = (spark.createDataFrame(resultados, schema=esquema)
              .withColumn("error_abs", F.abs(F.col("pct_gemini") - F.col("pct_real")))
              .withColumn("modelo", F.lit(MODELO))
              .withColumn("version_prompt", F.lit(VERSION_PROMPT))
              .withColumn("fecha_validacion", F.current_timestamp()))
    # mergeSchema porque añado la columna nueva version_prompt a la tabla existente
    (nuevos.write.format("delta").mode("append")
           .option("mergeSchema", "true").saveAsTable(TABLA))

# Las filas antiguas tienen version_prompt vacío: eran la v1
v = (spark.table(TABLA).filter("pct_gemini IS NOT NULL")
     .withColumn("version_prompt", F.coalesce("version_prompt", F.lit("v1"))))

display(v.groupBy("modelo", "version_prompt").agg(
    F.count("*").alias("imagenes_ok"),
    F.round(F.avg("error_abs"), 2).alias("error_medio_abs_pp"),
    F.round(F.avg(F.col("pct_gemini") - F.col("pct_real")), 2).alias("sesgo_medio_pp"),
    F.round(F.avg((F.col("error_abs") < 5).cast("int")) * 100, 1).alias("pct_con_error_menor_5pp"),
).orderBy("modelo", "version_prompt"))

StatementMeta(, d122b542-f558-4572-b018-72c797b161e7, 35, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, aae2961d-14b4-41ef-8c75-69d807e5614c)

In [19]:
# Comparo los dos modelos solo en las imágenes que han validado ambos
v = spark.table(TABLA).filter("pct_gemini IS NOT NULL")
flash = v.filter("modelo = 'gemini-3.5-flash'").select("imagen", F.col("error_abs").alias("err_flash"))
lite = v.filter("modelo = 'gemini-3.5-flash-lite'").select("imagen", F.col("error_abs").alias("err_lite"))

display(flash.join(lite, "imagen").agg(
    F.count("*").alias("imagenes_comunes"),
    F.round(F.avg("err_flash"), 2).alias("error_medio_flash"),
    F.round(F.avg("err_lite"), 2).alias("error_medio_lite"),
))

# Sesgo de Flash Lite: si sale positivo, tiende a sobrestimar; si sale negativo, a quedarse corto
display(v.filter("modelo = 'gemini-3.5-flash-lite'").agg(
    F.round(F.avg(F.col("pct_gemini") - F.col("pct_real")), 2).alias("sesgo_medio_pp")
))

StatementMeta(, d122b542-f558-4572-b018-72c797b161e7, 36, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 49e02043-7327-4ae3-a0a0-508ee75defc0)

SynapseWidget(Synapse.DataFrame, a58a811c-76c5-40d1-a16f-b20ec0672283)